In [ ]:
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import math


from helpers import *
from loss_and_eval import *
from optim import *



class Layer:
    def __init__(self, input_neurons, output_neurons, softmax = False, output_layer = False):
        self.j = output_neurons
        self.k = input_neurons

        self.W = np.random.normal(scale = 1/math.sqrt(self.j), size = (self.j, self.k))
        self.b = np.random.normal(size = self.j)

        self.activations = None
        self.weighted_inputs = None

        self.softmax = softmax
        self.output_layer = output_layer

        self.clear_grad()

    def __call__(self, prev_activations):
        return self.forward(prev_activations)


    def forward(self, prev_activations):
        z = np.add(self.W @ prev_activations.reshape(self.k), self.b.reshape)

        if self.softmax: a = np_softmax(z)
        else: a = np_sigmoid(z)

        self.activations = a
        self.weighted_inputs = z

        return a
        


    def backward(self, prev_activations, y = None, next_W_T = None, next_dZ = None):
        if self.output_layer:
            dZ = np.subtract(self.activations, y.reshape(-1, 1)) # BP1, BCE, NLL

        else:
            dZ = (next_W_T @ next_dZ) * np_dsigmoid(self.weighted_inputs) # BP2
            
        # BP3
        dW = dZ @ prev_activations.reshape(self.k, 1).T


        # BP4
        self.dW = np.add(self.dW, dW)
        self.dB = np.add(self.dB, dZ)

        return dZ


    def clear_grad(self):
        self.dW = np.zeros(shape = (self.j, self.k))
        self.dB = np.zeros(shape = self.j)



class Convolution:
    def __init__(self, filters, channels, kernel_size, input_size, stride = 1):
        self.filters = filters
        self.channels = channels
        self.kernel_size = kernel_size
        self.input_size = input_size
        self.stride = stride

        self.output_size = (self.input_size - self.kernel_size) // self.stride + 1

 
        self.W = np.random.normal(
            size = (self.filters, self.channels, self.kernel_size, self.kernel_size)
        )
        self.b = np.random.normal(
            size = self.filters
        )

        self.weighted_inputs = None
        self.clear_grad()


    def __call__(self, x):
            return self.forward(x)

    
    def clear_grad(self):
        self.dW = np.zeros(shape = (self.filters, self.channels, self.kernel_size, self.kernel_size))
        self.dB = np.zeros(shape = self.filters)

    

    def forward(self, x):
        # x: (channels, length, width)
        
        f_maps = np.zeros(shape=(self.filters, self.output_size, self.output_size))
        weighted_inputs = np.zeros_like(f_maps)

        # hard-coded
        for f in range(self.filters):
            f_map = np.zeros(shape=(self.output_size, self.output_size))
            weighted_input = np.zeros_like(f_map)

            for i in range(self.output_size):
                for j in range(self.output_size):
                    z = self.b[f]

                    for u in range(self.kernel_size):
                        for v in range(self.kernel_size):
                            for c in range(self.channels):
                                input_channel = x[c]
                                W = self.W[f, c, :, :] # (kernel_size, kernel_size)

                                row = self.stride * i + u
                                col = self.stride * j + v
                                z += input_channel[row][col] * W[u][v]


                    f_map[i][j] = sigmoid(z)
                    weighted_input[i][j] = z

            f_maps[f] = f_map
            weighted_inputs[f] = weighted_input

        self.weighted_inputs = weighted_inputs

        return f_maps

        # matrix implementation


    def backward(self, prev_activations, dA):

        dZ = np.zeros(shape = (self.filters, self.output_size, self.output_size))
        prev_dA = np.zeros(shape = (self.channels, self.input_size, self.input_size))


        for f in range(self.filters):
            for i in range(self.output_size):
                for j in range(self.output_size):
                    dZ[f][i][j] = dA[f][i][j] * dsigmoid(self.weighted_inputs[f][i][j]) # calculcate dZ
                    self.dB[f] += dZ[f][i][j] # update bias gradient

                    for u in range(self.kernel_size):
                        for v in range(self.kernel_size):
                            for c in range(self.channels): 
                                W = self.W[f, c, :, :]
                                row = self.stride * i + u
                                col = self.stride * j + v

                                self.dW[f][c][u][v] += dZ[f][i][j] * prev_activations[c][row][col] # update weight gradient
                                prev_dA[c][row][col] += dZ[f][i][j] * W[u][v] # calculate dA for the previous layer


        return prev_dA

    

         


class MaxPool:
    def __init__(self, f_maps, input_size, pool_size):
        self.f_maps = f_maps
        self.input_size = input_size
        self.pool_size = pool_size

        assert self.input_size % self.pool_size == 0, f"Pool size {self.pool_size} is not valid for {self.input_size}x{self.input_size} feature maps"
        self.output_size = self.input_size // self.pool_size

        self.dZ_prev_dA = np.zeros(shape = (self.f_maps, self.input_size, self.input_size))


    def __call__(self, f_maps):
        return self.forward(f_maps)

    def forward(self, f_maps):
        pooled_f_maps = np.zeros(shape = (self.f_maps, self.output_size, self.output_size))

        for f in range(self.f_maps):
            f_map = f_maps[f]
            pooled_f_map = np.zeros(shape = (self.output_size, self.output_size))

            for i in range(self.output_size):
                for j in range(self.output_size):

                    a = float("-inf")
                    max_i, max_j = 0, 0

                    for u in range(self.pool_size):
                        for v in range(self.pool_size):
                            row = i * self.pool_size + u
                            col = j * self.pool_size + v

                            if f_map[row][col] > a:
                                a = f_map[row][col]
                                max_i, max_j = row, col

                    pooled_f_map[i][j] = a
                    self.dZ_prev_dA[f][max_i][max_j] = 1.0
                    


            pooled_f_maps[f] = pooled_f_map

        return pooled_f_maps

    

    def backward(self, next_W_T, next_dZ):
        dZ = np.reshape(next_W_T @ next_dZ, (self.f_maps, self.output_size, self.output_size))
        prev_dA = np.zeros(shape = (self.f_maps, self.input_size, self.input_size)) 

        for f in range(self.f_maps):
            for i in range(self.output_size):
                for j in range(self.output_size):
                    for u in range(self.pool_size):
                        for v in range(self.pool_size):
                            row = i * self.pool_size + u
                            col = j * self.pool_size + v

                            prev_dA[f][row][col] = dZ[f][i][j] * self.dZ_prev_dA[f][row][col] # calc dA of previous layer

        return prev_dA
        
                    



class CNN:
    def __init__(self, filters_list):

        self.conv1 = Convolution(filters=filters_list[0], channels=1, kernel_size=5, input_size=28)
        self.pool1 = MaxPool(f_maps=filters_list[0], input_size=self.conv1.output_size, pool_size=2)
        
        self.layer1 = Layer(
            input_neurons=self.pool1.f_maps * self.pool1.output_size ** 2,
            output_neurons=100,
        )
        self.classifier = Layer(
            input_neurons=100, output_neurons=10,
            output_layer=True, softmax=True
        )

        self.layers = [self.conv1, self.pool1, self.layer1, self.classifier]
        self.x = None

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        self.activations_list = []
        self.x = x
        
        output = x
        for layer in self.layers:
            output = layer(output)
            self.activations_list.append(output)

        return output

    def backward(self, y):
        L = len(self.layers)
        for l in range(L-1, -1, -1): # go from L-1 to 0
            layer = self.layers[l]

            if (l-1 < 0): prev_activations = self.x
            else: prev_activations = self.activations_list[l-1]

            if type(layer) == Layer: 
                if layer.output_layer:
                    next_dZ = layer.backward(prev_activations, y=y)
                else:
                    next_dZ = layer.backward(prev_activations, next_W_T=self.layers[l+1].W.T, next_dZ=next_dZ)

            if type(layer) == MaxPool:
                prev_dA = layer.backward(self.layers[l+1].W.T, next_dZ)

            if type(layer) == Convolution:
                prev_dA = layer.backward(prev_activations, prev_dA)

        
    def step(self, optim):
        for layer in self.layers:
            if type(layer) != MaxPool: optim(layer)
    
    def clear_grad(self):
        for layer in self.layers:
            if type(layer) != MaxPool: layer.clear_grad()
        






In [2]:
np.random.seed(42)


train = MNIST(root="data", train=True, download=True)
test = MNIST(root="data", train=False, download=True)

x_train, y_train = [], []
x_test, y_test = [], []

for image, label in train:
    x = np.reshape([pixel / 255.0 for pixel in image.getdata()], shape=(1, 28, 28))
    y = np.zeros(shape=10)
    y[label] = 1.0

    x_train.append(x)
    y_train.append(y)


for image, label in test:
    x = np.reshape([pixel / 255.0 for pixel in image.getdata()], shape=(1, 28, 28))
    y = np.zeros(shape=10)
    y[label] = 1.0

    x_test.append(x)
    y_test.append(y)

x_val = x_train[50000:]
y_val = y_train[50000:]

x_train = x_train[:50000]
y_train = y_train[:50000]

In [3]:
model = CNN([20])
batch_size = 10
epochs = 100
lr = 0.1
weight_decay = 0

optim = Optimizer(lr=lr, batch_size=batch_size, weight_decay=weight_decay)

train_losses = []
val_accs = []
epochs_list = []


for i in range(epochs):
    num_train_samples = len(x_train)
    train_loss = 0.0

    combined = list(zip(x_train, y_train))
    np.random.shuffle(combined)
    x_train, y_train = zip(*combined)

    pbar = tqdm(total=num_train_samples, unit="samples")

    for batch_index in range(0, num_train_samples, batch_size):
        for offset in range(batch_size):
            s = batch_index + offset
            if (s >= num_train_samples): break

            x, y = x_train[s], y_train[s]
            preds = model(x)
            model.backward(y) 

            loss = NLL(y, preds)
            train_loss += loss

            # if s % 1024 == 0: print(loss)
            pbar.update(1)
            # if (s % 1000 == 0): print(f"{s}/{num_samples}")

        model.step(optim)
        model.clear_grad()


    pbar.close()

    train_loss += L2(model, optim) # L2 Regularization
    train_loss /= num_train_samples

    correct = 0.0
    num_val_samples = len(x_val)

    for s in range(num_val_samples):
        x, y = x_val[s], y_val[s]

        preds = model(x)
        correct += acc(y, preds)    

    val_acc = 100 * correct / num_val_samples   
        
    train_losses.append(train_loss)
    val_accs.append(val_acc)
    epochs_list.append(i+1)

    
    print(f"Epoch: {i+1} | Loss: {train_loss:.4f} | Acc: {val_acc:.2f}%")  

    plt.clf()
    plt.style.use("dark_background")
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.axis([1, epochs, 0, max(train_losses)])
    plt.plot(epochs_list, train_losses, c = "white")
    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")

    plt.subplot(1, 2, 2)
    plt.axis([1, epochs, min(val_accs), 100])
    plt.plot(epochs_list, val_accs, c = "white")
    plt.xlabel("Epoch")
    plt.ylabel("Val Acc")

    plt.show()





  0%|          | 0/50000 [00:00<?, ?samples/s]

(20, 1, 5, 5) (20,)
(20, 1, 5, 5) (20,)
(100, 2880) (100,)
(100, 2880) (100, 100)
(10, 100) (10,)
(10, 100) (10, 10)
[  7.23796238  -2.28912114  -0.20394216  -5.69855167   7.44048293
   4.41491463   7.58566964   3.11262984  -4.7157413   -4.77870246
  -3.57734846  -4.47838823  -6.84215845   5.4614006    4.05748928
   3.48885642  -0.57793004  -4.22230762 -10.44671355  -5.53533495
   2.34998589  -3.38272922   2.35117679  -2.17409837  -6.18824903
  -6.01590561   3.60919527  -2.82367021   4.81189635   3.08949035
  -0.76333537   2.626544    -3.51710776  -4.52326076  -3.82577358
  -5.74437134   6.22263411   0.49782759   4.90454145   2.767441
  -6.37078795  -3.72403948  -5.41426531   2.33174555 -11.97060138
  -2.5846514    3.8009758   -2.3894547   -2.31737127  -3.05846334
  -0.20591032   5.04751333   1.95737354   9.77863202  -3.91806457
  -4.03755917  -0.16694861   1.27667445  -8.33329416   4.98160146
  -3.64574829   2.03426142   1.22235656  -5.42118301  -1.06831658
  -4.83759887   5.57847626 

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'